In [2]:
import redis

# 连接你本地启动的 Redis
r = redis.Redis(host='localhost', port=6379, decode_responses=True)

# 测试连接
print(r.ping())  # 输出 True 表示连接成功

True


In [3]:
import json

def save_message(user_id, role, content):
    """将一条对话消息存入 Redis List"""
    key = f"chat:{user_id}"
    message = {"role": role, "content": content}
    # 将字典转为 JSON 字符串后存入，保持结构清晰
    r.rpush(key, json.dumps(message, ensure_ascii=False))

def get_history(user_id, limit=10):
    """取出最近 N 条对话记录"""
    key = f"chat:{user_id}"
    # -limit 到 -1 表示从倒数第 limit 条取到最后一条
    raw_messages = r.lrange(key, -limit, -1)
    return [json.loads(m) for m in raw_messages]

# 模拟对话
save_message("user_1001", "user", "你好，我想查一下订单")
save_message("user_1001", "assistant", "好的，请提供订单号")
save_message("user_1001", "user", "订单号是 12345")

# 读取最近 2 条
history = get_history("user_1001", limit=2)
for msg in history:
    print(f"{msg['role']}: {msg['content']}")
# 输出:
# user: 订单号是 12345
# assistant: 好的，请提供订单号 (注意：这里演示的是取最后2条，实际顺序取决于你读取的逻辑)

assistant: 好的，请提供订单号
user: 订单号是 12345


In [4]:
from kafka import KafkaProducer
import json

# 1. 创建生产者
# bootstrap_servers 指向你的 Kafka 地址，默认端口 9092
producer = KafkaProducer(
    bootstrap_servers=['localhost:9092'],
    # 使用 value_serializer 自动将字典转为 JSON 字节流，省去手动 dumps 和 encode
    value_serializer=lambda v: json.dumps(v, ensure_ascii=False).encode('utf-8')
)

# 2. 准备要发送的消息（例如一条对话记录）
message = {
    "user_id": "user_1001",
    "role": "user",
    "content": "你好，我想查一下订单"
}

# 3. 发送消息到指定 Topic
future = producer.send('chat-history', value=message)

# 4. （可选）阻塞等待，确认发送结果
try:
    record_metadata = future.get(timeout=10)
    print(f"消息已发送至 Topic: {record_metadata.topic}, 分区: {record_metadata.partition}, 偏移量: {record_metadata.offset}")
except Exception as e:
    print(f"发送失败: {e}")

# 5. 关闭生产者前，刷新缓冲区确保所有消息都已发出
producer.flush()
producer.close()

消息已发送至 Topic: chat-history, 分区: 0, 偏移量: 0


In [ ]:
from kafka import KafkaConsumer
import json

# 1. 创建消费者
consumer = KafkaConsumer(
    'chat-history',  # 订阅的 Topic
    bootstrap_servers=['localhost:9092'],
    group_id='chat-processor-group',  # 消费组 ID，自定义
    # 反序列化：将收到的字节流自动转为 Python 字典
    value_deserializer=lambda m: json.loads(m.decode('utf-8')),
    # 如果组内没有历史 offset，则从最早的消息开始读
    auto_offset_reset='earliest',
    # 自动提交消费位移（默认开启）
    enable_auto_commit=True
)

print("开始监听消息...")
# 2. 循环消费
for message in consumer:
    # message.value 已经是解析好的字典
    data = message.value
    print(f"收到消息 | 分区: {message.partition} | 偏移量: {message.offset}")
    print(f"内容: {data}")
    
    # 在这里可以加入你的业务逻辑，比如写入数据库或调用其他服务

/tmp/ipykernel_936/1629255713.py:5: DeprecationWarning: value_deserializer does not implement kafka.serializer.Deserializer
  consumer = KafkaConsumer(


开始监听消息...
收到消息 | 分区: 0 | 偏移量: 0
内容: {'user_id': 'user_1001', 'role': 'user', 'content': '你好，我想查一下订单'}
